In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import pandas as pd
df = pd.read_csv('Iris.csv')
df.head()

In [ ]:
import zipfile
import io

with zipfile.ZipFile(io.BytesIO(uploaded['archive.zip']), 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
df = pd.read_csv('/content/Iris.csv')

task 2 ml lab

In [ ]:
from copy import deepcopy

DOMAINS = [
    ["Sunny", "Rainy", "Overcast"],
    ["Hot", "Mild", "Cool"],
    ["High", "Normal"],
    ["Weak", "Strong"]
]

QUESTION = "?"
NULL = "∅"

def covers(hypothesis, example):
    for h, x in zip(hypothesis, example):
        if h == NULL:
            return False
        if h != QUESTION and h != x:
            return False
    return True

def more_general_or_equal(h1, h2):
    more_general_parts = []
    for a, b in zip(h1, h2):
        mg = (a == QUESTION) or (a == b)
        mg = mg or (a != NULL and b == NULL)
        more_general_parts.append(mg)
    return all(more_general_parts)

def minimal_generalizations(h, x):
    new_h = list(h)
    for i, (hi, xi) in enumerate(zip(h, x)):
        if hi == NULL:
            new_h[i] = xi
        elif hi != QUESTION and hi != xi:
            new_h[i] = QUESTION
    return [new_h]

def minimal_specializations(h, x, domains):
    specializations = []
    for i, hi in enumerate(h):
        if hi == QUESTION:
            for val in domains[i]:
                if val != x[i]:
                    new_h = list(h)
                    new_h[i] = val
                    specializations.append(new_h)
        elif hi != NULL:
            if hi == x[i]:
                pass
    return specializations

def remove_more_specific(G):
    G2 = []
    for g in G:
        dominated = False
        for g2 in G:
            if g != g2 and more_general_or_equal(g2, g):
                if g2 != g:
                    dominated = True
                    break
        if not dominated:
            G2.append(g)
    return G2

def candidate_elimination(examples, domains):
    n_attrs = len(domains)
    S = [NULL] * n_attrs
    G = [[QUESTION] * n_attrs]

    print("Initial S:", S)
    print("Initial G:", G)
    print("-" * 60)

    for idx, (x, label) in enumerate(examples, start=1):
        print(f"Example {idx}: {x} -> {label}")

        if label == "Yes":
            G = [g for g in G if covers(g, x)]
            if not covers(S, x):
                gens = minimal_generalizations(S, x)
                S = gens[0]
            G = [g for g in G if more_general_or_equal(g, S)]

        else:
            if covers(S, x):
                S = [NULL] * n_attrs
            new_G = []
            for g in G:
                if covers(g, x):
                    specs = minimal_specializations(g, x, domains)
                    specs = [s for s in specs if more_general_or_equal(s, S)]
                    new_G.extend(specs)
                else:
                    new_G.append(g)
            G = new_G
            G = remove_more_specific(G)

        print("S:", S)
        print("G:", G)
        print("-" * 60)

    print("Final S (specific boundary):", S)
    print("Final G (general boundary):")
    for g in G:
        print(" ", g)
    return S, G

if __name__ == "__main__":
    dataset = [
        (["Sunny", "Hot", "High", "Weak"], "No"),
        (["Sunny", "Hot", "High", "Strong"], "No"),
        (["Overcast", "Hot", "High", "Weak"], "Yes"),
        (["Rainy", "Mild", "High", "Weak"], "Yes"),
        (["Rainy", "Cool", "Normal", "Weak"], "Yes"),
        (["Rainy", "Cool", "Normal", "Strong"], "No"),
        (["Overcast", "Cool", "Normal", "Strong"], "Yes")
    ]

    candidate_elimination(dataset, DOMAINS)
    print("Updated Specific Hypothesis:", specific_h)
    print("Updated General Hypothesis:", general_h)
    print("-" * 30)


print("\nFinal Specific Hypothesis (S):")
print(specific_h)
print("\nFinal General Hypotheses (G):")
# Manually set the final G based on the analysis above
final_general_h = [('Overcast', '?', '?', '?'), ('?', 'Mild', '?', '?'), ('?', '?', 'High', '?')]
print(final_general_h)
